# Lab 07 — 00 Source Preparation

Downloads Chicago data, injects controlled quality fixtures, and creates canonical snapshot versions.

In [ ]:
from pathlib import Path
import sys
cwd=Path.cwd().resolve()
project_root=next((p for p in [cwd,*cwd.parents] if (p/'src'/'lab07').exists()),None)
if project_root and str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
if project_root and str(project_root/'tools') not in sys.path: sys.path.insert(0,str(project_root/'tools'))
dbutils.widgets.text('catalog','dbr_dev','01 Catalog'); dbutils.widgets.text('schema','parvinbadalov','02 Schema'); dbutils.widgets.text('volume_name','lab07_data_quality','03 Volume'); dbutils.widgets.text('run_id','manual','04 Run ID')
catalog=dbutils.widgets.get('catalog'); schema=dbutils.widgets.get('schema'); volume_name=dbutils.widgets.get('volume_name'); run_id=dbutils.widgets.get('run_id')
assert catalog=='dbr_dev' and schema=='parvinbadalov', f'Lab 07 requires dbr_dev.parvinbadalov, got {catalog}.{schema}'
volume_root=f'/Volumes/{catalog}/{schema}/{volume_name}'


In [ ]:
from license_batch_loader import download
from pyspark.sql import functions as F
dbutils.widgets.text('source_from_date','2024-01-01','05 Source From'); dbutils.widgets.text('source_max_rows','300000','06 Max Rows'); dbutils.widgets.text('snapshot_cutoffs','2024-12-31,2025-12-31,2026-08-15','07 Snapshot Cutoffs')
source_from_date=dbutils.widgets.get('source_from_date'); source_max_rows=int(dbutils.widgets.get('source_max_rows')); cutoffs=[x.strip() for x in dbutils.widgets.get('snapshot_cutoffs').split(',') if x.strip()]
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_name}")
landing=f'{volume_root}/landing/events'; dbutils.fs.rm(landing,True); dbutils.fs.mkdirs(landing)
manifest=download(landing,source_from_date=source_from_date,max_rows=source_max_rows)
print(manifest)


In [ ]:
from lab07.transformations import prepare_business_licenses
from lab07.quality_rules import classify_license_records
from lab07.snapshot_policy import canonical_snapshot, assert_unique_snapshot
raw=spark.read.json(f'{volume_root}/landing/events/*.json')
assert raw.count()>0
# Add deterministic DQ fixtures that are easy to distinguish from real source rows.
sample=raw.filter('id IS NOT NULL AND license_number IS NOT NULL').limit(4).collect()
fixtures=[]
if len(sample)>=4:
    def one(r): return spark.createDataFrame([r],schema=raw.schema)
    fixtures=[one(sample[0]).withColumn('id',F.lit('LAB07_TEST_BAD_STATUS')).withColumn('license_status',F.lit('BAD')).withColumn('_fixture_kind',F.lit('INVALID_LICENSE_STATUS')),
              one(sample[1]).withColumn('id',F.lit('LAB07_TEST_BAD_ZIP')).withColumn('zip_code',F.lit('ABC')).withColumn('_fixture_kind',F.lit('INVALID_ZIP')),
              one(sample[2]).withColumn('id',F.lit('LAB07_TEST_WARN_DBA')).withColumn('doing_business_as_name',F.lit(None).cast('string')).withColumn('_fixture_kind',F.lit('WARN_DBA_MISSING')),
              one(sample[3]).withColumn('id',F.lit('LAB07_TEST_BAD_DATES')).withColumn('license_start_date',F.lit('2026-12-31T00:00:00')).withColumn('expiration_date',F.lit('2026-01-01T00:00:00')).withColumn('_fixture_kind',F.lit('EXPIRATION_BEFORE_START'))]
landed=raw
for f in fixtures: landed=landed.unionByName(f,allowMissingColumns=True)
landed.write.mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{catalog}.{schema}.business_license_landing')
classified=classify_license_records(prepare_business_licenses(landed)); source=classified.filter("_dq_status <> 'QUARANTINE' AND _fixture_kind IS NULL")
frames=[]
for version,cutoff in enumerate(cutoffs,1):
    snap=canonical_snapshot(source,cutoff); assert_unique_snapshot(snap); frames.append(snap.withColumn('snapshot_version',F.lit(version)).withColumn('snapshot_cutoff',F.to_timestamp(F.lit(cutoff))))
feed=frames[0]
for f in frames[1:]: feed=feed.unionByName(f,allowMissingColumns=True)
feed.write.mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{catalog}.{schema}.business_license_snapshot_feed')
display(feed.groupBy('snapshot_version').agg(F.count('*').alias('rows'),F.countDistinct('license_number').alias('distinct_keys')))
print('SOURCE PREPARATION COMPLETE')
